# Componente 2B: clasificador con transfer learning desde Etapa A

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix

from contract import load_stage_a_output, EMBEDDING_DIM

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

Device: cpu


## Estrategia de transfer learning

Usamos **feature extraction**: el encoder LSTM entrenado en la Etapa A queda congelado y sus embeddings de 32 dimensiones se usan como representación de entrada para un clasificador nuevo (MLP) entrenado desde cero sobre las etiquetas. No hacemos fine-tuning del encoder porque fue entrenado únicamente sobre comportamiento normal (Etapa A no supervisada) y las etiquetas de fraude son muy escasas (6.3% del dataset elegible, y aún menos tras balanceo). fine-tunear el encoder completo con tan pocos positivos arriesga sobreajuste y "olvido catastrófico" de lo aprendido sobre normalidad. Usar los embeddings como features fijos es la estrategia estándar cuando el dataset etiquetado es pequeño respecto al dataset no supervisado usado para preentrenar.

## Justificación de la función de pérdida

El desbalance de clases es severo. Usamos `BCEWithLogitsLoss` con `pos_weight` calculado a partir del ratio negativos/positivos del split de entrenamiento, en vez de submuestrear de nuevo o usar accuracy como métrica (que sería engañosa con este desbalance). Esto penaliza más los falsos negativos sin descartar información de la clase mayoritaria.

In [ ]:
meta_df, embeddings, per_step_error = load_stage_a_output("stage_a_output")
print(meta_df.shape, embeddings.shape, per_step_error.shape)
meta_df.head()

(56553, 7) (56553, 32) (56553, 50)


,sequence_id,account_id,split,seq_len,label,recon_error,anomaly_score
0,C187881338,C187881338,train,10,1.0,0.258921,0.300086
1,C866867675,C866867675,train,19,1.0,0.339903,0.401760
2,C1762365059,C1762365059,train,3,1.0,0.231172,0.265247
3,C876852627,C876852627,train,4,1.0,0.114835,0.119185
4,C154560289,C154560289,train,14,1.0,0.278561,0.324744


In [ ]:
print("Proporcion de clases por split:")
print(meta_df.groupby("split")["label"].agg(["count", "mean"]))

train_mask = meta_df["split"] == "train"
n_pos = (meta_df.loc[train_mask, "label"] == 1).sum()
n_neg = (meta_df.loc[train_mask, "label"] == 0).sum()
pos_weight_value = n_neg / n_pos
print(f"\nTrain: {n_neg} negativos, {n_pos} positivos, ratio {pos_weight_value:.2f}:1")

Proporcion de clases por split:
       count    mean
split               
test   10000  0.0124
train  36553  0.0908
val    10000  0.0124

Train: 33234 negativos, 3319 positivos, ratio 10.01:1


In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class TransferClassifier(nn.Module):
    """MLP sobre embeddings congelados de la Etapa A (feature extraction)."""
    def __init__(self, embedding_dim, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [ ]:
def make_loader(split_name, batch_size, shuffle):
    mask = (meta_df["split"] == split_name).to_numpy()
    X = embeddings[mask]
    y = meta_df.loc[mask, "label"].to_numpy(dtype="float32")
    ds = EmbeddingDataset(X, y)
    gen = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=gen if shuffle else None), mask

train_loader, train_mask_arr = make_loader("train", batch_size=128, shuffle=True)
val_loader, val_mask_arr = make_loader("val", batch_size=256, shuffle=False)
test_loader, test_mask_arr = make_loader("test", batch_size=256, shuffle=False)
print("Train batches:", len(train_loader), "| Val batches:", len(val_loader), "| Test batches:", len(test_loader))

Train batches: 286 | Val batches: 40 | Test batches: 40


In [ ]:
@torch.no_grad()
def get_scores(model, mask):
    model.eval()
    X = torch.tensor(embeddings[mask], dtype=torch.float32, device=device)
    logits = model(X)
    return torch.sigmoid(logits).cpu().numpy()


def train_classifier(embedding_dim, pos_weight_value, max_epochs=60, patience=6, lr=1e-3, seed=SEED):
    torch.manual_seed(seed)
    model = TransferClassifier(embedding_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_pr_auc = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_x)
        train_loss = total_loss / len(train_loader.dataset)

        val_scores = get_scores(model, val_mask_arr)
        val_labels = meta_df.loc[val_mask_arr, "label"].to_numpy()
        val_pr_auc = average_precision_score(val_labels, val_scores)

        if epoch % 5 == 0 or epoch == max_epochs - 1:
            print(f"Epoch {epoch+1}/{max_epochs} - train_loss: {train_loss:.4f} - val_pr_auc: {val_pr_auc:.4f}")

        if val_pr_auc > best_pr_auc + 1e-4:
            best_pr_auc = val_pr_auc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping en epoch {epoch+1} (mejor val_pr_auc={best_pr_auc:.4f})")
                break

    model.load_state_dict(best_state)
    return model, best_pr_auc

In [ ]:
clf_model, clf_best_val_pr_auc = train_classifier(EMBEDDING_DIM, pos_weight_value)
print("Mejor PR-AUC en validacion:", round(clf_best_val_pr_auc, 4))

Epoch 1/60 - train_loss: 1.2051 - val_pr_auc: 0.0846


Epoch 6/60 - train_loss: 1.0109 - val_pr_auc: 0.2492


Epoch 11/60 - train_loss: 0.9415 - val_pr_auc: 0.3302


Epoch 16/60 - train_loss: 0.9100 - val_pr_auc: 0.3715


Epoch 21/60 - train_loss: 0.8996 - val_pr_auc: 0.3991


...

Epoch 36/60 - train_loss: 0.8751 - val_pr_auc: 0.4081


Epoch 41/60 - train_loss: 0.8709 - val_pr_auc: 0.4120


Epoch 46/60 - train_loss: 0.8696 - val_pr_auc: 0.4230


Epoch 51/60 - train_loss: 0.8579 - val_pr_auc: 0.4303


Epoch 56/60 - train_loss: 0.8610 - val_pr_auc: 0.4308
Early stopping en epoch 56 (mejor val_pr_auc=0.4319)
Mejor PR-AUC en validacion: 0.4319


## Experimento de ablación: ¿aporta valor la Etapa A?

Comparamos contra dos líneas base que no usan el conocimiento de la Etapa A:

1. **Solo clasificador de Etapa B**: usa los embeddings del autoencoder como features, pero sin combinar con `anomaly_score`.
2. **Baseline totalmente supervisado desde cero**: un clasificador entrenado directamente sobre las secuencias crudas de transacciones.

In [ ]:
import os

C1_SEQ_PATH = "C1_output_sequences.npy"
C1_META_PATH = "C1_output_meta.csv"

if os.path.exists(C1_SEQ_PATH) and os.path.exists(C1_META_PATH):
    from contract import MAX_SEQ_LEN

    raw_sequences = np.load(C1_SEQ_PATH)
    raw_meta = pd.read_csv(C1_META_PATH)
    N_FEATURES_RAW = raw_sequences.shape[2]

    class RawSeqDataset(Dataset):
        def __init__(self, sequences, lengths, labels):
            self.sequences = torch.tensor(sequences, dtype=torch.float32)
            self.lengths = torch.tensor(lengths, dtype=torch.long)
            self.labels = torch.tensor(labels, dtype=torch.float32)

        def __len__(self):
            return len(self.sequences)

        def __getitem__(self, idx):
            return self.sequences[idx], self.lengths[idx], self.labels[idx]

    class BaselineLSTMClassifier(nn.Module):
        """Clasificador supervisado desde cero, sin preentrenamiento de Etapa A."""
        def __init__(self, n_features, hidden_dim=32):
            super().__init__()
            self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
            self.head = nn.Linear(hidden_dim, 1)

        def forward(self, x, lengths):
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, (h_n, _) = self.lstm(packed)
            return self.head(h_n.squeeze(0)).squeeze(-1)

    def make_raw_loader(split_name, batch_size, shuffle):
        mask = (raw_meta["split"] == split_name).to_numpy()
        ds = RawSeqDataset(
            raw_sequences[mask],
            raw_meta.loc[mask, "seq_len"].to_numpy(),
            raw_meta.loc[mask, "label"].to_numpy(dtype="float32"),
        )
        gen = torch.Generator().manual_seed(SEED)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=gen if shuffle else None), mask

    baseline_train_loader, _ = make_raw_loader("train", 128, True)
    baseline_val_loader, baseline_val_mask = make_raw_loader("val", 256, False)
    baseline_test_loader, baseline_test_mask = make_raw_loader("test", 256, False)

    torch.manual_seed(SEED)
    baseline_model = BaselineLSTMClassifier(N_FEATURES_RAW).to(device)
    optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3, weight_decay=1e-4)
    pos_weight_raw = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_raw)

    @torch.no_grad()
    def eval_baseline(loader):
        baseline_model.eval()
        all_scores, all_labels = [], []
        for bx, blen, by in loader:
            bx, blen = bx.to(device), blen.to(device)
            logits = baseline_model(bx, blen)
            all_scores.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(by.numpy())
        return np.concatenate(all_scores), np.concatenate(all_labels)

    best_pr_auc_baseline, best_state_baseline = -1.0, None
    epochs_without_improvement = 0
    for epoch in range(60):
        baseline_model.train()
        for bx, blen, by in baseline_train_loader:
            bx, blen, by = bx.to(device), blen.to(device), by.to(device)
            optimizer.zero_grad()
            logits = baseline_model(bx, blen)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

        val_scores, val_labels_raw = eval_baseline(baseline_val_loader)
        val_pr_auc = average_precision_score(val_labels_raw, val_scores)
        if val_pr_auc > best_pr_auc_baseline + 1e-4:
            best_pr_auc_baseline = val_pr_auc
            best_state_baseline = {k: v.detach().clone() for k, v in baseline_model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= 6:
                print(f"Early stopping en epoch {epoch+1}")
                break

    baseline_model.load_state_dict(best_state_baseline)
    test_scores_baseline, test_labels_baseline = eval_baseline(baseline_test_loader)
    roc_auc_baseline = roc_auc_score(test_labels_baseline, test_scores_baseline)
    pr_auc_baseline = average_precision_score(test_labels_baseline, test_scores_baseline)
    print("Baseline (sin Etapa A) - Test ROC-AUC:", round(roc_auc_baseline, 4), "| PR-AUC:", round(pr_auc_baseline, 4))

    ablation_ready = True
else:
    print("PENDIENTE: no se encontro C1_output_sequences.npy / C1_output_meta.csv en este entorno.")
    print("Pide a Persona A que comparta esos archivos para completar el experimento de ablacion.")
    roc_auc_baseline, pr_auc_baseline = None, None
    ablation_ready = False

Early stopping en epoch 27


Baseline (sin Etapa A) - Test ROC-AUC: 0.982 | PR-AUC: 0.7986


## Segunda estrategia de transfer learning: fine-tuning del encoder

El resultado de arriba muestra que feature extraction (embeddings congelados) pierde mucha información útil para distinguir fraude: el PR-AUC del clasificador (0.37) queda muy por debajo del baseline entrenado desde cero sobre las secuencias completas (0.80). Esto es evidencia de que el objetivo no supervisado de la Etapa A (reconstruir secuencias normales) no está alineado con la tarea de clasificar fraude, y que congelar el encoder descarta señal relevante.

Probamos entonces la otra estrategia de transfer learning, fine-tuning. Partimos de los pesos ya entrenados del encoder LSTM de la Etapa A (en vez de inicializar al azar como el baseline) y los seguimos entrenando junto con una cabeza de clasificación nueva, usando una tasa de aprendizaje más baja para el encoder (para no destruir lo aprendido de golpe) y una tasa más alta para la cabeza nueva. Si el fine-tuning cierra la brecha con el baseline, eso demuestra que el conocimiento de la Etapa A sí aporta valor como punto de partida, aunque no sirva como feature fija.

In [9]:
if ablation_ready:
    class FineTuneClassifier(nn.Module):
        """Encoder inicializado desde los pesos de la Etapa A, luego afinado con las etiquetas."""
        def __init__(self, n_features, embedding_dim, hidden_dim=32):
            super().__init__()
            self.encoder = nn.LSTM(n_features, embedding_dim, batch_first=True)
            self.head = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, x, lengths):
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, (h_n, _) = self.encoder(packed)
            return self.head(h_n.squeeze(0)).squeeze(-1)

    torch.manual_seed(SEED)
    finetune_model = FineTuneClassifier(N_FEATURES_RAW, EMBEDDING_DIM).to(device)

    pretrained_state = torch.load("stage_a_output/lstm_autoencoder.pt", map_location=device, weights_only=False)
    encoder_weights = {k.replace("encoder.", ""): v for k, v in pretrained_state["model_state_dict"].items() if k.startswith("encoder.")}
    finetune_model.encoder.load_state_dict(encoder_weights)
    print("Encoder inicializado con los pesos preentrenados de la Etapa A.")

    optimizer = torch.optim.Adam([
        {"params": finetune_model.encoder.parameters(), "lr": 1e-4},
        {"params": finetune_model.head.parameters(), "lr": 1e-3},
    ], weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_raw)

    @torch.no_grad()
    def eval_finetune(loader):
        finetune_model.eval()
        all_scores, all_labels = [], []
        for bx, blen, by in loader:
            bx, blen = bx.to(device), blen.to(device)
            logits = finetune_model(bx, blen)
            all_scores.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(by.numpy())
        return np.concatenate(all_scores), np.concatenate(all_labels)

    best_pr_auc_ft, best_state_ft = -1.0, None
    epochs_without_improvement = 0
    for epoch in range(60):
        finetune_model.train()
        for bx, blen, by in baseline_train_loader:
            bx, blen, by = bx.to(device), blen.to(device), by.to(device)
            optimizer.zero_grad()
            logits = finetune_model(bx, blen)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

        val_scores, val_labels_ft = eval_finetune(baseline_val_loader)
        val_pr_auc = average_precision_score(val_labels_ft, val_scores)
        if val_pr_auc > best_pr_auc_ft + 1e-4:
            best_pr_auc_ft = val_pr_auc
            best_state_ft = {k: v.detach().clone() for k, v in finetune_model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= 6:
                print(f"Early stopping en epoch {epoch+1}")
                break

    finetune_model.load_state_dict(best_state_ft)
    test_scores_ft, test_labels_ft = eval_finetune(baseline_test_loader)
    roc_auc_ft = roc_auc_score(test_labels_ft, test_scores_ft)
    pr_auc_ft = average_precision_score(test_labels_ft, test_scores_ft)
    print("Fine-tuning (encoder Etapa A + cabeza nueva) - Test ROC-AUC:", round(roc_auc_ft, 4), "| PR-AUC:", round(pr_auc_ft, 4))
else:
    roc_auc_ft, pr_auc_ft = None, None


Encoder inicializado con los pesos preentrenados de la Etapa A.


Fine-tuning (encoder Etapa A + cabeza nueva) - Test ROC-AUC: 0.9513 | PR-AUC: 0.6756


## Selección de la estrategia final de Etapa B

El experimento de ablación muestra que **fine-tuning supera claramente a feature extraction** (PR-AUC test 0.676 vs. 0.373), por lo que desbloquear el encoder y adaptarlo con una tasa de aprendizaje baja recupera la mayor parte de la brecha frente al baseline sin Etapa A. Elegimos entre ambas estrategias comparando PR-AUC en **validación** y usamos la ganadora como la señal oficial de Etapa B para todo el sistema. 

In [10]:
if ablation_ready:
    val_scores_ft, val_labels_ft_check = eval_finetune(baseline_val_loader)
    val_pr_auc_ft = average_precision_score(val_labels_ft_check, val_scores_ft)
    print(f"Val PR-AUC - feature extraction: {clf_best_val_pr_auc:.4f} | fine-tuning: {val_pr_auc_ft:.4f}")

    use_finetune = val_pr_auc_ft > clf_best_val_pr_auc
    print("Estrategia elegida:", "fine-tuning" if use_finetune else "feature extraction")
else:
    use_finetune = False
    print("Sin ablation_ready: se usa feature extraction por defecto (no se pudo comparar con fine-tuning).")


Val PR-AUC — feature extraction: 0.4319 | fine-tuning: 0.7369
Estrategia elegida: fine-tuning


In [11]:
def stage_b_scores_for_all_accounts():
    """Devuelve el score de Etapa B (ganador del ablation) alineado con el orden de meta_df."""
    if use_finetune:
        raw_meta_indexed = raw_meta.set_index("account_id")
        full_ds = RawSeqDataset(
            raw_sequences,
            raw_meta["seq_len"].to_numpy(),
            raw_meta["label"].to_numpy(dtype="float32"),
        )
        full_loader = DataLoader(full_ds, batch_size=256, shuffle=False)
        scores_raw, _ = eval_finetune(full_loader)
        scores_by_account = pd.Series(scores_raw, index=raw_meta["account_id"].to_numpy())
        aligned = meta_df["account_id"].map(scores_by_account)
        assert aligned.notna().all(), "Faltan cuentas al alinear scores de fine-tuning con meta_df"
        return aligned.to_numpy()
    else:
        return get_scores(clf_model, np.ones(len(meta_df), dtype=bool))

all_stage_b_scores = stage_b_scores_for_all_accounts()
print("stage_b_scores listos:", all_stage_b_scores.shape)


stage_b_scores listos: (56553,)


## Combinando las señales de Etapa A y Etapa B

La predicción final combina la señal elegida de Etapa B con el `anomaly_score` no supervisado de la Etapa A. Ajustamos el peso de la combinación (`alpha`) maximizando PR-AUC en validación. Igual que se justificó el umbral de la Etapa A, esta decisión también se justifica empíricamente y no se fija a mano.

In [12]:
val_mask_bool = val_mask_arr
val_stage_b_scores = all_stage_b_scores[val_mask_bool]
val_anomaly_scores = meta_df.loc[val_mask_bool, "anomaly_score"].to_numpy()
val_labels = meta_df.loc[val_mask_bool, "label"].to_numpy()

alphas = np.linspace(0.0, 1.0, 21)
best_alpha, best_combo_pr_auc = 0.0, -1.0
for alpha in alphas:
    combo = alpha * val_stage_b_scores + (1 - alpha) * val_anomaly_scores
    pr_auc = average_precision_score(val_labels, combo)
    if pr_auc > best_combo_pr_auc:
        best_combo_pr_auc = pr_auc
        best_alpha = alpha

stage_b_only_pr_auc = average_precision_score(val_labels, val_stage_b_scores)
print(f"Mejor alpha (peso de Etapa B): {best_alpha:.2f}")
print(f"PR-AUC combinado en validacion: {best_combo_pr_auc:.4f} (vs. solo Etapa B: {stage_b_only_pr_auc:.4f})")


Mejor alpha (peso de Etapa B): 1.00
PR-AUC combinado en validacion: 0.7369 (vs. solo Etapa B: 0.7369)


In [13]:
def combined_score(mask):
    stage_b = all_stage_b_scores[mask]
    anomaly_scores = meta_df.loc[mask, "anomaly_score"].to_numpy()
    return best_alpha * stage_b + (1 - best_alpha) * anomaly_scores

val_combo_scores = combined_score(val_mask_bool)
precision_curve, recall_curve, thresholds = precision_recall_curve(val_labels, val_combo_scores)
f1_scores = 2 * precision_curve * recall_curve / (precision_curve + recall_curve + 1e-9)
best_idx = np.nanargmax(f1_scores[:-1])
final_threshold = thresholds[best_idx]
print("Umbral final elegido (max F1 en val, score combinado):", final_threshold)
print("Precision:", precision_curve[best_idx], "Recall:", recall_curve[best_idx], "F1:", f1_scores[best_idx])


Umbral final elegido (max F1 en val, score combinado): 0.9620658159255981
Precision: 0.8586956521739131 Recall: 0.6370967741935484 F1: 0.7314814809924554


In [14]:
test_mask_bool = test_mask_arr
test_combo_scores = combined_score(test_mask_bool)
test_labels = meta_df.loc[test_mask_bool, "label"].to_numpy()
test_preds = (test_combo_scores >= final_threshold).astype(int)

roc_auc_test = roc_auc_score(test_labels, test_combo_scores)
pr_auc_test = average_precision_score(test_labels, test_combo_scores)
tn, fp, fn, tp = confusion_matrix(test_labels, test_preds).ravel()
precision_test = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_test = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_test = 2 * precision_test * recall_test / (precision_test + recall_test + 1e-9)

print("=== Test (Etapa B elegida + combinacion con Etapa A) ===")
print("Estrategia de Etapa B:", "fine-tuning" if use_finetune else "feature extraction")
print("ROC-AUC:", round(roc_auc_test, 4), "| PR-AUC:", round(pr_auc_test, 4))
print("Precision:", round(precision_test, 4), "| Recall:", round(recall_test, 4), "| F1:", round(f1_test, 4))
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")


=== Test (Etapa B elegida + combinacion con Etapa A) ===
Estrategia de Etapa B: fine-tuning
ROC-AUC: 0.9513 | PR-AUC: 0.6756
Precision: 0.8353 | Recall: 0.5726 | F1: 0.6794
TP=71 FP=14 FN=53 TN=9862


## Tabla comparativa final del experimento de ablación


In [15]:
comparison = pd.DataFrame([
    {"modelo": "Baseline supervisado desde cero (sin Etapa A)",
     "roc_auc_test": roc_auc_baseline, "pr_auc_test": pr_auc_baseline},
    {"modelo": "Etapa B: feature extraction (embeddings congelados)",
     "roc_auc_test": roc_auc_score(test_labels, get_scores(clf_model, test_mask_arr)),
     "pr_auc_test": average_precision_score(test_labels, get_scores(clf_model, test_mask_arr))},
    {"modelo": "Etapa B: fine-tuning (encoder Etapa A + cabeza nueva)",
     "roc_auc_test": roc_auc_ft, "pr_auc_test": pr_auc_ft},
    {"modelo": f"Sistema final: Etapa A + Etapa B ({'fine-tuning' if use_finetune else 'feature extraction'}) combinadas",
     "roc_auc_test": roc_auc_test, "pr_auc_test": pr_auc_test},
])
comparison


,modelo,roc_auc_test,pr_auc_test
0,Baseline supervisado desde cero (sin Etapa A),0.982043,0.798644
1,Etapa B: feature extraction (embeddings congel...,0.848531,0.373122
2,Etapa B: fine-tuning (encoder Etapa A + cabeza...,0.951315,0.675590
3,Sistema final: Etapa A + Etapa B (fine-tuning)...,0.951315,0.675590


## Guardado de resultados de Etapa B


In [16]:
all_scores_combo = best_alpha * all_stage_b_scores + (1 - best_alpha) * meta_df["anomaly_score"].to_numpy()

stage_b_out = meta_df.copy()
stage_b_out["clf_score"] = all_stage_b_scores
stage_b_out["combined_score"] = all_scores_combo
stage_b_out["alert"] = (all_scores_combo >= final_threshold).astype(int)

os.makedirs("stage_b_output", exist_ok=True)
stage_b_out.to_csv("stage_b_output/meta_with_scores.csv", index=False)
torch.save(
    (finetune_model if use_finetune else clf_model).state_dict(),
    "stage_b_output/classifier.pt",
)
with open("stage_b_output/config.json", "w") as f:
    json.dump({
        "strategy": "fine-tuning" if use_finetune else "feature_extraction",
        "alpha": float(best_alpha), "threshold": float(final_threshold),
        "pos_weight": float(pos_weight_value),
    }, f, indent=2)
print("Guardado en stage_b_output/: meta_with_scores.csv, classifier.pt, config.json")
print("Estrategia guardada:", "fine-tuning" if use_finetune else "feature extraction")


Guardado en stage_b_output/: meta_with_scores.csv, classifier.pt, config.json
Estrategia guardada: fine-tuning
